# Project 2: Credit Card Fraud Detection (Binary Classification)

**Course:** Machine Learning, Chapter 14 Capstone Project
**Author:** Mozammel Hoque Sagor

## Objective
Decide for each card transaction whether it is **fraud (1)** or **legitimate (0)**. This is a **binary classification** problem with **extreme class imbalance**: fraud is a tiny fraction of all transactions.

## Dataset
Kaggle "Credit Card Fraud Detection" CSV (anonymised European card transactions). Expected columns:
`Time, V1 ... V28, Amount, Class` (`V1`-`V28` are anonymised PCA components, `Class` is the label).

Put the CSV in the `data/` folder (default name `/data/creditcard.csv`) or change `DATA_PATH` below.

## Pipeline (matches the capstone guidelines)
| Step | Section |
|---|---|
| 1. Data acquisition & EDA | 1, 2 |
| 2. Preprocessing (scaling, imbalance handling) | 3, 4 |
| 3. Model selection: baseline + complex model | 4, 5 |
| 4. Hyperparameter tuning (RandomizedSearchCV) | 5 |
| 5. Rigorous evaluation (Recall, F1, PR-AUC, Confusion Matrix) | 6, 7 |

## Two rules this notebook follows
1. **No data leakage:** the train/test split happens first, scalers are fitted on training data only, and **SMOTE is applied only to training data** (inside a pipeline).
2. **Accuracy is the wrong metric here.** A model that always says "not fraud" is ~99.8% accurate and completely useless. We focus on **Recall** (how many frauds we catch), **Precision**, **F1** and **PR-AUC**.


## 0. Setup

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
os.makedirs("figures", exist_ok=True)


def save_fig(name):
    """Save the current figure to figures/ (handy for the PDF report)."""
    plt.tight_layout()
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")


In [ ]:
# ---- Settings you can change ----
DATA_PATH = os.environ.get("DATA_PATH", "/data/creditcard.csv")  # path to the CSV file
TEST_SIZE = 0.2    # 20% of the data is kept for the final test
N_ITER = 10        # number of random-search trials
CV_FOLDS = 3       # cross-validation folds


## 1. Data acquisition

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)

required = ["Time", "Amount", "Class"] + [f"V{i}" for i in range(1, 29)]
absent = [c for c in required if c not in df.columns]
if absent:
    raise ValueError(f"Missing columns {absent}. Columns found: {list(df.columns)}")

display(df.head())
print("\nMissing values in total:", int(df.isna().sum().sum()))
display(df[["Time", "Amount", "Class"]].describe().T)


FileNotFoundError: [Errno 2] No such file or directory: '/data/creditcard.csv'

## 2. Exploratory data analysis (EDA)

In [ ]:
counts = df["Class"].value_counts().sort_index()
print(counts.rename({0: "Legitimate (0)", 1: "Fraud (1)"}))
print(f"\nFraud share: {df['Class'].mean():.4%}")

# Hour of day (the Time column is seconds since the first transaction)
df["Hour"] = ((df["Time"] // 3600) % 24).astype(int)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

sns.countplot(x="Class", data=df, ax=ax[0])
ax[0].set_yscale("log")
ax[0].set_title("Class balance (log scale)")

sns.boxplot(x="Class", y=np.log1p(df["Amount"]), data=df, ax=ax[1])
ax[1].set_ylabel("log(1 + Amount)")
ax[1].set_title("Transaction amount by class")

df.groupby("Hour")["Class"].mean().mul(100).plot(kind="bar", ax=ax[2])
ax[2].set_ylabel("Fraud rate (%)")
ax[2].set_title("Fraud rate by hour of day")

save_fig("fraud_eda_overview")
plt.show()


In [ ]:
corr = df.drop(columns=["Class"]).corrwith(df["Class"])
top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(12)
top.sort_values().plot(kind="barh", figsize=(7, 5), title="Features most correlated with fraud")
save_fig("fraud_top_correlations")
plt.show()


### EDA insights
- *Insight 1:* ...The data is extremely imbalanced. Only 492 of 284,807 transactions (0.173%) are fraud, so a model that always predicts "not fraud" would be about 99.83% accurate while catching no fraud. Because of this I judge the models with precision, recall, F1 and PR-AUC instead of accuracy.
- *Insight 2:* ...Transaction amount alone does not separate fraud from normal transactions. On the log scale the median fraud amount is lower, but the two distributions overlap heavily.
- *Insight 3:* ...The fraud rate looks highest around hours 2 and 4 of the day. The data covers only about two days and there are few night-time transactions, so this pattern is only suggestive and may be noise.

## 3. Preprocessing
- **Duplicates** are removed *before* splitting, so identical rows cannot end up in both train and test (that would be leakage).
- `V1`-`V28` are already PCA-transformed, so only `Amount` and `Hour` need scaling. We use `RobustScaler` because `Amount` has big outliers.
- The **stratified split** keeps the same tiny fraud share in train and test.

In [ ]:
n_dups = int(df.duplicated().sum())
df = df.drop_duplicates()
print(f"Removed {n_dups:,} duplicate rows")

X = df.drop(columns=["Class", "Time"])   # Hour replaces the raw Time column
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print(f"Fraud in train: {int(y_train.sum())} ({y_train.mean():.4%}) | in test: {int(y_test.sum())} ({y_test.mean():.4%})")

SCALE_COLS = ["Amount", "Hour"]
FEATURE_NAMES = SCALE_COLS + [c for c in X.columns if c not in SCALE_COLS]   # order after the transformer


def make_preprocessor():
    # fitted on training data only, because it lives inside each pipeline
    return ColumnTransformer([("scale", RobustScaler(), SCALE_COLS)], remainder="passthrough")


## 4. Baselines
First a **dummy** model (always predicts "not fraud") to show why accuracy is misleading, then a **Logistic Regression** baseline using two different ways of handling imbalance:
- `class_weight="balanced"`: mistakes on the rare class cost more
- **SMOTE**: creates synthetic fraud examples, **in the training data only**

In [ ]:
def evaluate(name, y_true, y_pred, y_score):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_score),
        "PR-AUC": average_precision_score(y_true, y_score),
    }


results, preds = [], {}   # preds[name] = (predicted labels, fraud scores)


def record(name, y_pred, y_score):
    results.append(evaluate(name, y_test, y_pred, y_score))
    preds[name] = (np.asarray(y_pred), np.asarray(y_score))


# 1) Dummy: always "not fraud"
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
record("Dummy (always not-fraud)", dummy.predict(X_test), dummy.predict_proba(X_test)[:, 1])

# 2) Logistic Regression with class weights
lr_bal = Pipeline([
    ("prep", make_preprocessor()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
]).fit(X_train, y_train)
record("LogReg (class_weight)", lr_bal.predict(X_test), lr_bal.predict_proba(X_test)[:, 1])

# 3) Logistic Regression with SMOTE (imblearn pipeline: SMOTE only touches the training data)
lr_smote = ImbPipeline([
    ("prep", make_preprocessor()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
]).fit(X_train, y_train)
record("LogReg (SMOTE)", lr_smote.predict(X_test), lr_smote.predict_proba(X_test)[:, 1])

display(pd.DataFrame(results).round(4))


Look at the **Dummy** row: very high accuracy but **Recall = 0**. That is why we judge the models by Recall, Precision, F1 and PR-AUC instead.

## 5. Complex model: XGBoost + hyperparameter tuning
XGBoost builds many small trees one after another, each fixing the mistakes of the previous ones. We handle imbalance with `scale_pos_weight` (how much more a fraud counts than a normal transaction) and tune it together with the other settings.

The search uses **stratified cross-validation on the training data** and optimises **PR-AUC** (`average_precision`), which suits rare-event problems better than accuracy.

In [ ]:
neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
spw = neg / pos
print(f"Legit: {neg:,} | Fraud: {pos:,} | ratio = {spw:.1f}")

xgb_pipe = Pipeline([
    ("prep", make_preprocessor()),
    ("clf", XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr",
        tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
    )),
])

param_dist = {
    "clf__n_estimators": [200, 300, 500],
    "clf__max_depth": [3, 4, 6, 8],
    "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
    "clf__subsample": [0.7, 0.85, 1.0],
    "clf__colsample_bytree": [0.6, 0.8, 1.0],
    "clf__min_child_weight": [1, 3, 5],
    "clf__scale_pos_weight": [1.0, float(np.sqrt(spw)), float(spw)],
}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    xgb_pipe, param_distributions=param_dist, n_iter=N_ITER,
    scoring="average_precision", cv=cv, random_state=RANDOM_STATE,
    n_jobs=1, verbose=1,
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print(f"Best cross-validated PR-AUC: {search.best_score_:.4f}")

best_xgb = search.best_estimator_
xgb_score = best_xgb.predict_proba(X_test)[:, 1]
record("XGBoost (tuned)", (xgb_score >= 0.5).astype(int), xgb_score)
display(pd.DataFrame(results).round(4))


### Choosing the decision threshold (without touching the test set)
By default a transaction is flagged when the fraud probability is at least 0.5, but that is not always the best cut-off. We pick the threshold that maximises **F1 on out-of-fold training predictions**, so the test set stays untouched.

*Business note:* if missing a fraud is much worse than a false alarm, you can choose a lower threshold to get higher Recall (and accept lower Precision).

In [ ]:
oof_score = cross_val_predict(best_xgb, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

prec, rec, thr = precision_recall_curve(y_train, oof_score)
f1_curve = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_thr = float(thr[np.argmax(f1_curve)])
print(f"Chosen threshold: {best_thr:.3f} (out-of-fold F1 = {f1_curve.max():.3f})")

record("XGBoost (tuned + threshold)", (xgb_score >= best_thr).astype(int), xgb_score)


NameError: name 'best_xgb' is not defined

## 6. Rigorous evaluation
- **Recall**: of all real frauds, how many did we catch? (the most important number here)
- **Precision**: of all transactions we flagged, how many were really fraud?
- **F1**: balance of Precision and Recall
- **PR-AUC**: quality of the fraud scores across all thresholds (the best single number for imbalanced data)

In [ ]:
results_df = pd.DataFrame(results).round(4)
display(results_df)

names = ["LogReg (class_weight)", "LogReg (SMOTE)", "XGBoost (tuned)", "XGBoost (tuned + threshold)"]
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, name in zip(axes.ravel(), names):
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds[name][0], display_labels=["Legit", "Fraud"],
        ax=ax, colorbar=False, cmap="Blues",
    )
    ax.set_title(name)
    ax.grid(False)
save_fig("fraud_confusion_matrices")
plt.show()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
plt.figure(figsize=(8, 6))
for name in names:
    p, r, _ = precision_recall_curve(y_test, preds[name][1])
    ap = average_precision_score(y_test, preds[name][1])
    plt.plot(r, p, label=f"{name} (PR-AUC {ap:.3f})")
plt.axhline(y_test.mean(), color="gray", linestyle="--", label="random guess")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curves (test set)")
plt.legend(loc="lower left")
save_fig("fraud_pr_curves")
plt.show()


In [ ]:
xgb_model = best_xgb.named_steps["clf"]
imp = pd.Series(xgb_model.feature_importances_, index=FEATURE_NAMES).sort_values().tail(15)
imp.plot(kind="barh", figsize=(7, 6), title="XGBoost feature importance (top 15)")
save_fig("fraud_feature_importance")
plt.show()


## 7. Business impact (for the presentation)

In [ ]:
final_name = "XGBoost (tuned + threshold)"
y_hat = preds[final_name][0]
tn, fp, fn, tp = confusion_matrix(y_test, y_hat).ravel()

print(f"Model: {final_name}")
print(f"Frauds in the test set:       {tp + fn}")
print(f"  caught (true positives):    {tp}  ({tp / (tp + fn):.1%} recall)")
print(f"  missed (false negatives):   {fn}")
print(f"False alarms (legit flagged): {fp} out of {tn + fp:,} legitimate transactions ({fp / (tn + fp):.3%})")
print()
print(classification_report(y_test, y_hat, target_names=["Legit", "Fraud"], digits=4))


## 8. Conclusion
- **Which model was best, and why?** The tuned XGBoost with the chosen threshold (0.840) was best on the test set: precision 0.986, recall 0.758, F1 0.857 and PR-AUC 0.817. Logistic Regression reached a PR-AUC of only about 0.67, and the Dummy model had 99.83% accuracy but 0 recall, which shows why accuracy is a poor metric here.
- **Trade-off:** XGBoost caught 72 of the 95 frauds in the test set and missed 23, with only 1 false alarm among 56,651 legitimate transactions. Logistic Regression caught more fraud (83 of 95) but raised 1,414 (class weights) and 1,482 (SMOTE) false alarms, about 17 for every fraud caught.
- **Limitations:** The test set contains only 95 frauds, so small differences (for example 73 vs 72 frauds caught) can be noise. The data covers about two days and was split randomly instead of by time. The features are anonymised, only 10 tuning settings were tried, and the threshold maximises F1 rather than real costs.
- **Future work:** Choose the threshold from real fraud losses and review costs, evaluate with a time-based split and repeated cross-validation, try more models and a wider search, and deploy the model with FastAPI.